# Week 6: Exercise 10 - ACP Server

**Goal:** Expose your agent as an HTTP service.

No API key needed for this exercise.


## Step 1: Data Storage


In [1]:
import json, uuid
from http.server import HTTPServer, BaseHTTPRequestHandler

AGENTS = {}
RUNS = {}


## Step 2: Implement the Request Handler

`ACPRequestHandler` serves three endpoints:

- `GET /agents` — list registered agents
- `GET /runs/{id}` — fetch one run by ID (404 if unknown)
- `POST /runs` — submit `{"input": "..."}`, creates and returns a run

Reference implementation below — study it, then reset to stubs and redo.


In [2]:
class ACPRequestHandler(BaseHTTPRequestHandler):
    """HTTP handler for ACP endpoints."""

    def _send_json(self, data, status=200):
        """TODO: Send a JSON response."""
        # Hint: json.dumps(data).encode(), send_response, headers, write body
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", f"{len(body)}")
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        """TODO: Handle GET /agents and GET /runs/{id}."""
        # Hint: check self.path == "/agents" for the agent list,
        #       self.path.startswith("/runs/") then split("/")[-1] for the run ID
        if self.path == "/agents":
            self._send_json({"agents": list(AGENTS.values())}, 200)

        elif self.path.startswith("/runs/"):
            run_id = self.path.split('/')[-1]
            run = RUNS.get(run_id)
            if not run:
                self._send_json({"error": f"Run not found"}, 404)
            else: self._send_json(run, 200)

        else: self._send_json({"error": "Wrong endpoint"}, 404)

    def do_POST(self):
        """TODO: Handle POST /runs."""
        # Hint: read Content-Length bytes from self.rfile, json.loads,
        #       create a run_id with str(uuid.uuid4())[:8], store it, respond 201
        if self.path == "/runs":
            content_len = int(self.headers.get("Content-Length", 0)) 
            data = json.loads(self.rfile.read(content_len))

            run_id = str(uuid.uuid4())[:8]
            RUNS[run_id] = {"id": run_id, "status": "completed", "result": f"Agent processed: {data.get('input', '')}"}

            self._send_json(RUNS[run_id], 201)

        else: self._send_json({"error": "Wrong endpoint"}, 404)




## Step 3: Register an Agent and Start the Server


In [3]:
AGENTS["agent-1"] = {
    "id": "agent-1",
    "name": "MiniHermes",
    "description": "A minimal agent for learning",
    "capabilities": ["chat", "tools"]
}

def run_server(port=8080):
    """Run the ACP server."""
    httpd = HTTPServer(("localhost", port), ACPRequestHandler)
    print(f"Server running at http://localhost:{port}")
    httpd.serve_forever()

# Uncomment after implementing the handler:
# run_server()


## Test Your Solution

Run the server in a background thread so the notebook doesn't block,
then hit the endpoints with urllib.


In [4]:
# Test 1: success flow (GET /agents, POST /runs, GET /runs/{id})
import threading, urllib.request, urllib.error

server = HTTPServer(("localhost", 8181), ACPRequestHandler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
print("Server running at http://localhost:8181")

# GET /agents -> should list agent-1
with urllib.request.urlopen("http://localhost:8181/agents") as resp:
    data = json.loads(resp.read())
    assert resp.status == 200, f"expected 200, got {resp.status}"
    assert any(a["id"] == "agent-1" for a in data["agents"]), "agent-1 missing"
print("  GET /agents OK:", [a["name"] for a in data["agents"]])

# POST /runs -> create a run
req = urllib.request.Request(
    "http://localhost:8181/runs",
    data=json.dumps({"input": "hello agent"}).encode(),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req) as resp:
    run = json.loads(resp.read())
    assert resp.status == 201, f"expected 201, got {resp.status}"
    assert run["status"] == "completed"
    assert "hello agent" in run["result"]
print("  POST /runs OK:", run)

# GET /runs/{id} -> fetch it back
with urllib.request.urlopen(f"http://localhost:8181/runs/{run['id']}") as resp:
    fetched = json.loads(resp.read())
    assert fetched["id"] == run["id"]
print("  GET /runs/{id} OK:", fetched["id"])

server.shutdown()
server.server_close()
print("Test 1 passed: success flow works.")


Server running at http://localhost:8181


127.0.0.1 - - [25/Aug/2026 15:16:28] "GET /agents HTTP/1.1" 200 -


  GET /agents OK: ['MiniHermes']


127.0.0.1 - - [25/Aug/2026 15:16:30] "POST /runs HTTP/1.1" 201 -


  POST /runs OK: {'id': '276db5fe', 'status': 'completed', 'result': 'Agent processed: hello agent'}


127.0.0.1 - - [25/Aug/2026 15:16:32] "GET /runs/276db5fe HTTP/1.1" 200 -


  GET /runs/{id} OK: 276db5fe
Test 1 passed: success flow works.


In [5]:
# Test 2: failure cases -> both should return 404
import threading, urllib.request, urllib.error, json
from http.server import HTTPServer

server = HTTPServer(("localhost", 8182), ACPRequestHandler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()

# Unknown path -> 404
try:
    urllib.request.urlopen("http://localhost:8182/nope")
    assert False, "should have raised HTTPError"
except urllib.error.HTTPError as e:
    assert e.code == 404, f"expected 404, got {e.code}"
print("  GET /nope -> 404 OK")

# Unknown run id -> 404 with error message
try:
    urllib.request.urlopen("http://localhost:8182/runs/does-not-exist")
    assert False, "should have raised HTTPError"
except urllib.error.HTTPError as e:
    assert e.code == 404, f"expected 404, got {e.code}"
    body = json.loads(e.read())
    assert body["error"] == "Run not found"
print("  GET /runs/does-not-exist -> 404 OK:", body)

server.shutdown()
server.server_close()
print("Test 2 passed: failure cases return 404.")


127.0.0.1 - - [25/Aug/2026 15:16:44] "GET /nope HTTP/1.1" 404 -


  GET /nope -> 404 OK


127.0.0.1 - - [25/Aug/2026 15:16:51] "GET /runs/does-not-exist HTTP/1.1" 404 -


  GET /runs/does-not-exist -> 404 OK: {'error': 'Run not found'}
Test 2 passed: failure cases return 404.


## Key Takeaways
- Exposing your agent over HTTP makes it accessible to other programs
- http.server is stdlib - no dependencies needed
